In [ ]:
import geopandas as gpd
from shapely.geometry import Point, LineString

In [ ]:
def remove_z_dims(_gdf: gpd.GeoDataFrame):
    _gdf.geometry = [
        (Point(g.coords[0][:2]) if len(g.coords[0]) > 2 else Point(g.coords[0]))
        if isinstance(g, Point)
        else (
            (LineString([c[:2] if len(c) > 2 else c for c in g.coords]))
            if isinstance(g, LineString)
            else Polygon([c[:2] if len(c) > 2 else c for c in g.exterior.coords])
        )
        for g in _gdf.geometry.values
    ]
    return _gdf

In [ ]:
gdf_rr_nodes = gpd.read_file("P:/5325/51036877_RR_unpaved_Oude_IJssel/300 Werkdocumenten/850_modellering/stap2RRinput/20260421/RR_input_ZOMER.gpkg").iloc[:10]
gdf_branches = gpd.read_file("C:/Users/NL1G6D/Desktop/open_projecten/Project_Roel/Nieuwe map/hydroobject.gpkg")

In [ ]:
gdf_branches = remove_z_dims(gdf_branches.explode())

In [ ]:
import geopandas as gpd
from shapely.ops import nearest_points
 

gdf_laterals = gdf_rr_nodes.copy()

# Build spatial index for lines
lines_sindex = gdf_branches.sindex
 
def project_point_to_nearest_line(point):
    # find candidate lines
    possible_matches_index = list(
        lines_sindex.nearest(point.bounds, 1)
    )
    nearest_line = gdf_branches.iloc[possible_matches_index[0]].geometry
 
    # project point onto line
    projected_point = nearest_line.interpolate(
        nearest_line.project(point)
    )
    return projected_point
 
gdf_laterals["geometry"] = gdf_rr_nodes.geometry.apply(
    project_point_to_nearest_line
)

In [ ]:
laterals = gpd.sjoin_nearest(
    gdf_rr_nodes,
    gdf_branches,
    how="left",
    distance_col="dist"
)

# laterals.apply(
#     lambda row: row.geometry.interpolate(
#         row.geometry.project(row.geometry)
#     ) if row.geometry else None,
#     axis=1
# )
laterals.plot()

In [ ]:
gdf = gpd.read_file("P:/5325/51036877_RR_unpaved_Oude_IJssel/300 Werkdocumenten/850_modellering/stap2RRinput/20260424/RR_input_ZOMER.gpkg")

In [ ]:
gdf.columns